# CRA Tutorial — Hands-on AI on the National Research Platform

This notebook is the hands-on portion of the tutorial. Everything runs
**inside this JupyterHub session** — no separate pods to launch and no YAML to
apply.

A one-line vocabulary primer, in case these are new:

- **LLM** — a large language model (the thing that answers prompts).
- **Inference** — running a prompt through a model to get an answer.
- **RAG** — *retrieval-augmented generation*: look up relevant text first,
  then ask the model to answer **using only that text**.

**What you'll do**

1. Verify the session has the env vars, `kubectl`, and (optionally) a GPU.
2. Call the **NRP managed LLM** (`https://ellm.nrp-nautilus.io/v1`) from Python.
3. Run a **local LLM** (Ollama + `mistral`) on the session's GPU, side by side.
4. Build a **RAG pipeline** over a real page of the NRP documentation, using
   NRP's **managed embedding model** — no vector database, nothing to download.

When you're done, [`3_opencode.md`](3_opencode.md) points an agentic coding CLI
at this same managed LLM.

**How to run a notebook (skip if you've used Jupyter before)**

- Grey cells are code; this one is text. Click a cell and press
  **Shift + Enter** to run it. Run them **top to bottom, in order**.
- `[*]` means a cell is still running; `[7]` means it finished. If stuck, use
  **Kernel → Restart Kernel** and re-run from the top.
- Cells that print `Skipping — no GPU` are fine on a CPU-only session.

**Prerequisites**

- Spawn with **1 × NVIDIA-A10 GPU, ~4 CPU cores, and ~16 GB RAM** to run the
  local-Ollama half of section 3 (the spawn form defaults to 1 core / 8 GB —
  too low; the `mistral` load needs ~4 GB). The managed-LLM and RAG sections
  also work on a CPU-only session.
- Use the `NRP Deep Learning & Data Science Full, PyTorch` image (default).
- The pod already exports `OPENAI_API_BASE` / `OPENAI_API_KEY` and has
  `kubectl` wired to the `nrp-training-k8s` namespace.

## 1. Setup check

Confirm the session has everything we'll need: the managed-LLM env vars,
`kubectl` against `nrp-training-k8s`, and `nvidia-smi` if you spawned with a
GPU. The Ollama sections will skip themselves gracefully if there's no GPU.


In [ ]:
import os, shutil, subprocess

print("OPENAI_API_BASE =", os.environ.get("OPENAI_API_BASE", "NOT SET"))
# Print the actual token. This is a shared workshop key — fine to show here;
# treat your own personal token as a secret.
key = os.environ.get("OPENAI_API_KEY", "")
print("OPENAI_API_KEY  =", key if key else "NOT SET")

kubectl = shutil.which("kubectl") or "/opt/conda/bin/kubectl"
print("kubectl path    =", kubectl)
print("kubectl version =", subprocess.run([kubectl, "version", "--client", "-o", "json"],
                                          capture_output=True, text=True).stdout[:120], "...")

print("can list pods in nrp-training-k8s:",
      subprocess.run([kubectl, "auth", "can-i", "list", "pods", "-n", "nrp-training-k8s"],
                     capture_output=True, text=True).stdout.strip())


In [ ]:
# GPU detection. Section 3 (local Ollama) need this; others are fine without.
import subprocess

HAS_GPU = False
try:
    out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=True).stdout.strip()
    print("GPU(s) detected:")
    print(out)
    HAS_GPU = True
except (FileNotFoundError, subprocess.CalledProcessError):
    print("No GPU in this session pod.")
    print("Section 3 (local Ollama) will skip the local-LLM half.")
    print("To run the full comparison, respawn with 1 x NVIDIA-A10 from the JupyterHub launcher.")


## 2. Option A — Use NRP's managed LLMs

There are **two ways to get an LLM in this notebook**, and they use the *same*
Python code:

- **Option A (this section): NRP's managed LLMs.** NRP runs a catalog of
  open-weights models for you behind one OpenAI-compatible URL — you send HTTP
  requests, run no pod, and no GPU is billed to you.
- **Option B (section 3): run your own LLM locally** with Ollama, on the GPU
  attached to this session.

You reach the managed service two ways: a **browser chat UI** at
[`https://nrp-openwebui.nrp-nautilus.io`](https://nrp-openwebui.nrp-nautilus.io),
and the **OpenAI-compatible API** at `https://ellm.nrp-nautilus.io/v1` (used
here). You authenticate with a **bearer token** — mint one at
[`https://nrp.ai/llmtoken`](https://nrp.ai/llmtoken); on this session it's
already exported as `OPENAI_API_KEY`.

**The catalog rotates** as the open-source frontier moves. Roughly, today:

| Model | Size | Good for |
|---|---|---|
| `qwen3` | 397B | frontier reasoning, largest context |
| `gpt-oss` | 120B | agentic tool-calling / code |
| `gemma` | 31B | general chat, multimodal |
| `kimi`, `glm-5`, `minimax-m2` | 230B–1T | coding (under evaluation) |
| `qwen3-embedding` | 8B | embeddings only (used for RAG below) |

> Some of these (`qwen3`, `minimax-m2`, `gpt-oss`, …) are **reasoning models**:
> they write a private chain-of-thought into a separate `reasoning` field
> before filling `content`, so they need a **larger `max_tokens`** or they hit
> the limit before answering. The `chat()` helper below handles that.

The same `openai` SDK code targets the managed endpoint, a local Ollama
server, or the OpenAI cloud — **just by changing `base_url`**. That portability
is the whole point of OpenAI-compatible APIs.

**NRP LLM docs:**
[overview](https://nrp.ai/documentation/userdocs/ai/llm-managed/) ·
[models](https://nrp.ai/documentation/userdocs/ai/llm-managed/models/) ·
[API access](https://nrp.ai/documentation/userdocs/ai/llm-managed/api-access/) ·
[client configs](https://nrp.ai/documentation/userdocs/ai/llm-managed/client-configs/)

In [ ]:
# List the models currently served by the NRP managed endpoint.
import os, json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ["OPENAI_API_BASE"])
models = client.models.list()
print(f"{len(models.data)} models available right now:\n")
for m in sorted(models.data, key=lambda x: x.id):
    print(f"  {m.id}")


In [ ]:
# A tiny helper we'll reuse across this notebook. The default model is
# `minimax-m2` (fast on NRP, and a good default). It's a *reasoning* model: it
# streams a private chain-of-thought into a separate `reasoning` field and only
# fills `content` once it concludes — so we default `max_tokens` high enough
# (1200) to leave room for both the thinking and the final answer. If a call
# still runs out of tokens mid-thought, we surface that reasoning rather than
# returning nothing.
def chat(prompt, model="minimax-m2", system=None, max_tokens=1200, llm=None):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    resp = (llm or client).chat.completions.create(
        model=model, messages=msgs, max_tokens=max_tokens, temperature=0.2,
    )
    msg = resp.choices[0].message
    if msg.content:
        return msg.content
    reasoning = getattr(msg, "reasoning", None) or getattr(msg, "reasoning_content", None)
    if reasoning:
        return f"(no final answer within max_tokens={max_tokens}; reasoning so far:)\n{reasoning}"
    return "(model produced no content — try raising max_tokens or another model)"

In [ ]:
# Single chat completion.
print(chat(
    "What is the National Research Platform?",
    system="Answer in one sentence.",
))


In [ ]:
# Switch to a smaller model — same code, just change the `model` arg.
print(chat("Explain Kubernetes namespaces in two sentences.", model="gemma-small"))

# Now a reasoning model. We give it room (max_tokens=800) so it finishes
# thinking AND produces a final answer — with too few tokens the whole budget
# is spent reasoning and `content` comes back empty (what the helper guards).
print("\n--- minimax-m2 (a reasoning model) ---")
print(chat("In one sentence, name a strength of this model.", model="minimax-m2", max_tokens=800))

# Try the others yourself — e.g. model="gpt-oss" (fast) or model="qwen3"
# (the 397B flagship; highest quality but slowest to respond).


## 3. Option B — Run your own LLM locally (Ollama on the session GPU)

Same OpenAI-compatible API, but now the model runs on **your** GPU inside this
JupyterHub session instead of on NRP's shared endpoint. We install Ollama,
pull `mistral` (7B Q4, ~4 GB), and hit it at `http://localhost:11434/v1` — then
ask the same prompt against NRP's managed `minimax-m2` and our local `mistral` side
by side. **Only `base_url` changes between the two.**

> These cells skip themselves if `HAS_GPU` is `False`. Respawn with
> **1 × NVIDIA-A10** to run the local half.

> **Heads-up on speed:** Ollama and its models install to fast local `/tmp`
> (re-downloaded each session, ~10 s) so the GPU is detected reliably. The
> first `mistral` call **pulls ~4 GB**, which takes a minute or two; after that
> it's loaded on the A10 and responds in a second or two.


In [ ]:
# Install Ollama onto the session's FAST local disk (/tmp), NOT the home
# directory. Ollama ships ~3.5 GB of CUDA libraries; the home volume is
# networked storage (Ceph), and reading 3.5 GB from it is too slow for Ollama's
# GPU probe — it times out and silently falls back to CPU. /tmp is node-local
# and fast, so the A10 is detected in under a second. (/tmp is wiped when the
# session stops, so this re-downloads each session — it only takes ~10 s.)
import os, shutil, subprocess

OLLAMA_DIR = "/tmp/ol"
OLLAMA_BIN = f"{OLLAMA_DIR}/bin/ollama"
OLLAMA_VERSION = "v0.24.0"  # pinned to a known-good release

if not HAS_GPU:
    print("Skipping Ollama install — no GPU in this session.")
elif os.path.exists(OLLAMA_BIN):
    print("Ollama already installed at", OLLAMA_BIN)
else:
    print(f"Downloading Ollama {OLLAMA_VERSION} to {OLLAMA_DIR} (~10 s)...")
    os.makedirs(OLLAMA_DIR, exist_ok=True)
    url = f"https://github.com/ollama/ollama/releases/download/{OLLAMA_VERSION}/ollama-linux-amd64.tar.zst"
    subprocess.run(
        f"curl -fsSL {url} | tar --use-compress-program=unzstd -xf - -C {OLLAMA_DIR}",
        shell=True, check=True,
    )
if HAS_GPU:
    os.environ["PATH"] = f"{OLLAMA_DIR}/bin" + os.pathsep + os.environ["PATH"]
    print("ollama binary at:", shutil.which("ollama"))


In [ ]:
# Start `ollama serve` in the background and wait until it answers.
#   - Everything (binary, CUDA libs, models) lives under /tmp/ol on fast local
#     disk, so the GPU probe finds the A10 in well under a second.
#   - OLLAMA_LLM_LIBRARY + LD_LIBRARY_PATH point at the bundled CUDA libraries.
#   - OLLAMA_MODELS keeps pulled models on /tmp too, so they load fast.
import os, socket, subprocess, time, urllib.request, urllib.error

def ollama_alive():
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3)
        return True
    except (urllib.error.URLError, ConnectionResetError, OSError, socket.timeout, TimeoutError):
        return False

LIB = f"{OLLAMA_DIR}/lib/ollama"

if not HAS_GPU:
    print("Skipping Ollama serve — no GPU in this session.")
elif ollama_alive():
    print("Ollama already running.")
else:
    print("Starting `ollama serve` in the background...")
    ollama_env = {
        **os.environ,
        "OLLAMA_HOST": "0.0.0.0:11434",
        "OLLAMA_LOAD_TIMEOUT": "15m",
        "OLLAMA_MODELS": f"{OLLAMA_DIR}/models",
        "OLLAMA_LIBRARY_PATH": LIB,
        "OLLAMA_LLM_LIBRARY": "cuda_v12",
        "LD_LIBRARY_PATH": ":".join([
            LIB, LIB + "/cuda_v12", "/usr/lib/x86_64-linux-gnu",
            os.environ.get("LD_LIBRARY_PATH", ""),
        ]),
    }
    subprocess.Popen(f"nohup {OLLAMA_BIN} serve > /tmp/ollama.log 2>&1 &", shell=True, env=ollama_env)
    t0 = time.time(); deadline = t0 + 120
    while time.time() < deadline:
        time.sleep(2)
        if ollama_alive():
            print(f"Ollama up after ~{int(time.time() - t0)}s.")
            break
    else:
        print("Ollama did not come up within 120s. Check /tmp/ollama.log")

# Report which device Ollama is using.
if HAS_GPU and ollama_alive():
    log = subprocess.run("grep -iE 'inference compute' /tmp/ollama.log | tail -1",
                         shell=True, capture_output=True, text=True).stdout.lower()
    if "library=cuda" in log:
        print("Device: GPU (CUDA) — the A10 was detected. \U0001f389")
    elif "id=cpu" in log:
        print("Device: CPU — GPU probe fell back to CPU; check /tmp/ollama.log")
    else:
        print("Device: unknown — see /tmp/ollama.log")


In [ ]:
# Pull the mistral 7B model (~4 GB). First time only.
import subprocess, time, json, urllib.request

def already_pulled(name="mistral"):
    try:
        r = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2))
        return any(name in m["name"] for m in r.get("models", []))
    except Exception:
        return False

if not HAS_GPU:
    print("Skipping — no GPU.")
elif already_pulled("mistral"):
    print("mistral already pulled.")
else:
    print("Pulling mistral... (~4 GB, 2-4 minutes the first time)")
    # Drive the pull via the HTTP API instead of `ollama pull`, because the CLI
    # writes \r-based progress bars that buffer indefinitely under nbconvert.
    body = json.dumps({"name": "mistral", "stream": True}).encode()
    req  = urllib.request.Request("http://localhost:11434/api/pull", data=body,
                                  headers={"Content-Type": "application/json"})
    t0 = time.time()
    last_status = None
    with urllib.request.urlopen(req) as resp:
        for raw in resp:
            try:
                evt = json.loads(raw.decode())
            except Exception:
                continue
            status = evt.get("status", "")
            # Only print on status transitions (not every kilobyte) to keep output tidy
            if status and status != last_status:
                print(f"  [{int(time.time()-t0):3d}s] {status}")
                last_status = status
            if evt.get("error"):
                raise RuntimeError(evt["error"])
    print(f"Done in {int(time.time()-t0)}s.")


In [ ]:
# Hello-world against the local model. Same OpenAI client, different base_url.
# Heads up: the first call after `ollama serve` started will be slow (~30-120s
# while llama.cpp loads the 4 GB of weights into VRAM). Subsequent calls in
# this notebook reuse the loaded model and respond in 1-2 seconds.
from openai import OpenAI

if HAS_GPU:
    local = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1",
                   timeout=900)  # generous timeout for the cold load
    print(chat(
        "Say hi from the GPU in this JupyterHub session.",
        model="mistral", llm=local, max_tokens=80,
    ))
else:
    print("Skipping — no GPU.")


In [ ]:
# Side-by-side: same prompt, managed minimax-m2 vs local mistral.
prompt = "Explain in two sentences why a research cluster might use Kubernetes namespaces."

print("=" * 78)
print("NRP MANAGED (minimax-m2)")
print("=" * 78)
print(chat(prompt, model="minimax-m2"))

if HAS_GPU:
    print()
    print("=" * 78)
    print("LOCAL (mistral, on this pod's GPU)")
    print("=" * 78)
    print(chat(prompt, model="mistral", llm=local))
else:
    print("\n(Local half skipped — no GPU.)")


**What to notice.** The Python code is byte-for-byte identical — same SDK,
same `chat.completions.create`, same messages. Only `base_url` differs.
Latency, answer style, and request privacy will differ:

| | Managed `minimax-m2` | Local `mistral` (this pod) |
|---|---|---|
| Where it runs | NRP GPUs you don't see | the A10 attached to this pod |
| Latency to first answer | thinks first, then streams | ~6 s (warm) |
| GPU billed | none | yours, while pod runs |
| Privacy | request leaves your namespace | never leaves your pod |
| Catalog | the NRP model list | anything `ollama pull` supports |

## 4. RAG — answer from NRP's own docs (no extra infrastructure)

**RAG** ("retrieval-augmented generation") makes an LLM answer from *your*
text instead of guessing: (1) split your docs into chunks and **embed** each
into a vector, (2) embed the question and find the closest chunks, (3) put
those chunks in the prompt with "answer only from this context."

The nice part on NRP: the **embedding model is managed too**. The same endpoint
that serves the chat models also serves `qwen3-embedding`, so we don't install
or download anything. And for a small corpus, a handful of vectors in NumPy is
all the "vector database" we need.

We'll pull **one real page of the NRP documentation** and answer questions from it.


In [ ]:
# Pull one real NRP documentation page (the cluster policies page) and clean it.
import requests, re

PAGE = "https://nrp.ai/documentation/userdocs/start/policies/"
RAW  = ("https://gitlab.nrp-nautilus.io/prp/nrp-site/-/raw/main/"
        "src/content/docs/Documentation/userdocs/start/policies.mdx")

md = requests.get(RAW, timeout=30).text
md = re.sub(r"^---.*?---\s*", "", md, flags=re.S)        # drop frontmatter
md = re.sub(r"^import .*$", "", md, flags=re.M)          # drop MDX imports
md = re.sub(r":::\w+(\[[^\]]*\])?|:::", "", md)          # drop admonition markers
md = re.sub(r"<[^>]+>", "", md)                          # drop stray HTML/JSX
md = re.sub(r"\n{3,}", "\n\n", md).strip()

# Split into ~700-char chunks. (A real corpus would chunk every page; one is plenty here.)
chunks = [md[i:i+700] for i in range(0, len(md), 550)]
print(f"Pulled {len(md)} chars from {PAGE}")
print(f"Split into {len(chunks)} chunks.")


In [ ]:
# Embed with NRP's MANAGED embedding model — no local model, nothing to download.
# Same `client`, same endpoint as the chat models; just a different call.
import numpy as np

def embed(texts):
    resp = client.embeddings.create(model="qwen3-embedding", input=texts)
    vecs = np.array([d.embedding for d in resp.data])
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)   # normalize for cosine

chunk_vecs = embed(chunks)          # one API call embeds the whole page
print(f"Embedded {len(chunks)} chunks into vectors of dim {chunk_vecs.shape[1]} (via qwen3-embedding).")

def retrieve(question, k=3):
    qv = embed([question])[0]
    scores = chunk_vecs @ qv         # cosine similarity (everything is normalized)
    top = scores.argsort()[-k:][::-1]
    return [(chunks[i], float(scores[i])) for i in top]


In [ ]:
# Answer a question by RAG. Same retrieval + embeddings; only model/llm change
# between the managed and local backends.
SYSTEM = ("Answer the question using ONLY the provided context. "
          "If the context doesn't contain the answer, say so. Be concise.")

def ask(question, model="minimax-m2", llm=None):
    context = "\n\n".join(text for text, _ in retrieve(question))
    return chat(f"Context:\n{context}\n\nQuestion: {question}", system=SYSTEM, model=model, llm=llm)

QUESTION = "Is it okay to run `sleep infinity` in a Job on NRP?"
print("Retrieved chunks:")
for text, score in retrieve(QUESTION):
    print(f"  score={score:.3f}  {text[:70].strip()}...")

print("\n--- Answer (NRP managed minimax-m2) ---")
print(ask(QUESTION))

if HAS_GPU:
    print("\n--- Answer (local mistral on the GPU) ---")
    print(ask(QUESTION, model="mistral", llm=local))
else:
    print("\n(Local answer skipped — no GPU. Same pipeline as the managed answer above.)")

**What to notice.**

- **Everything came from NRP's managed endpoint** — the chat model *and* the
  `qwen3-embedding` embedding model. No models were installed or downloaded.
- Retrieval was ~5 lines of NumPy. That's all a vector database does under the
  hood; for one page it's overkill. For a **large, persistent corpus** you'd
  swap the NumPy array for NRP's managed **Milvus** vector database — same
  three steps (embed, search, prompt), just scalable.
- Managed vs local was one argument: the same `ask()` answered through NRP's
  `minimax-m2` and the local `mistral`, with identical retrieval and embeddings.

## 5. Cleanup

Stopping the local Ollama process frees the GPU memory. (The RAG step used only
NRP's managed services and in-memory NumPy, so there's nothing else to tear down.)


In [ ]:
# Stop the local Ollama daemon (frees GPU memory).
import subprocess
subprocess.run(["pkill", "-x", "ollama"], check=False)
print("Ollama stopped.")


## Takeaways

- The same OpenAI-compatible Python code targeted two inference backends:
  NRP's managed `ellm.nrp-nautilus.io` and a local Ollama on this session's
  GPU. Only `base_url` changed.
- Managed LLMs are the lowest-friction path for classrooms — no token
  handoff, no GPU billed to students. Run your own locally when you need a
  specific model or want the data to never leave your pod.
- RAG is "embed your docs, retrieve the closest chunks, prompt the LLM with
  'answer from context only.'" Here the **embedding model (`qwen3-embedding`)
  and the LLM both came from NRP's managed endpoint** — no local models — and a
  few NumPy lines replaced a vector database for a one-page corpus.

**Next:** [`3_opencode.md`](3_opencode.md) — point an agentic coding CLI
(`opencode`) at this same managed LLM and have it write a small program.